# Feature Engineering

---

1. Import packages
2. Load data
3. Feature engineering

---

## 1. Import packages

In [1]:
import pandas as pd

---
## 2. Load data

In [28]:
df = pd.read_csv('./clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [3]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


---

## 3. Feature engineering

### Difference between off-peak prices in December and preceding January

Below is the code created by your colleague to calculate the feature described above. Use this code to re-create this feature and then think about ways to build on this feature to create features with a higher predictive power.

In [29]:
price_df = pd.read_csv('price_data (1).csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [30]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()
monthly_price_by_id
# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


In [31]:
df = pd.merge(df, diff, on='id')
df.head(3)


,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1,0.020057,3.700961
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0,-0.003767,0.177779
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0,-0.004670,0.177779


#### 1. Remove redundant columns

Two things stand out on inspection:
- `margin_gross_pow_ele` and `margin_net_pow_ele` are almost perfectly correlated (corr ≈ 0.9999,
  identical in 99.99% of rows) — keeping both adds noise rather than signal, so we drop the gross
  figure and keep net margin, the one that actually matters for profitability.
- `var_year_price_off_peak` / `_peak` / `_mid_peak` and their `var_6m_` equivalents turn out to be
  **exact sums** of the matching `_var` + `_fix` columns already in the data — they carry zero extra information over their components, so we drop the totals.


In [32]:
print("Correlation (gross vs net margin):", df['margin_gross_pow_ele'].corr(df['margin_net_pow_ele']))
print("Share of rows identical:", (df['margin_gross_pow_ele'] == df['margin_net_pow_ele']).mean())

redundant_totals = [
    'var_year_price_off_peak', 'var_year_price_peak', 'var_year_price_mid_peak',
    'var_6m_price_off_peak', 'var_6m_price_peak', 'var_6m_price_mid_peak',
]
print("Max diff vs var+fix sum:",
      (df['var_year_price_off_peak'] - (df['var_year_price_off_peak_var'] + df['var_year_price_off_peak_fix'])).abs().max())

df = df.drop(columns=['margin_gross_pow_ele'] + redundant_totals)


Correlation (gross vs net margin): 0.9999141087859935
Share of rows identical: 0.9998630699712447
Max diff vs var+fix sum: 5.684341886080802e-14


####  Expand the date columns

Raw dates aren't directly usable by most ML models. The dataset is a snapshot as of **January 2016**
(the earliest `date_end` is 2016-01-28), so we convert each date column into "months relative to
2016-01-01" - this keeps the information (tenure, time-to-renewal, etc.) in a numeric, model-ready form.


In [33]:
reference_date = pd.Timestamp('2016-01-01')

def months_between(later, earlier):
    return (later.dt.year - earlier.dt.year) * 12 + (later.dt.month - earlier.dt.month)

df['months_activ'] = months_between(pd.Series(reference_date, index=df.index), df['date_activ'])
df['months_to_end'] = months_between(df['date_end'], pd.Series(reference_date, index=df.index))
df['months_modif_prod'] = months_between(pd.Series(reference_date, index=df.index), df['date_modif_prod'])
df['months_renewal'] = months_between(pd.Series(reference_date, index=df.index), df['date_renewal'])

df = df.drop(columns=['date_activ', 'date_end', 'date_modif_prod', 'date_renewal'])
df[['months_activ', 'months_to_end', 'months_modif_prod', 'months_renewal']].describe()


,months_activ,months_to_end,months_modif_prod,months_renewal
count,14606.000000,14606.000000,14606.000000,14606.000000
mean,59.511091,6.426537,36.370327,5.816925
std,19.352836,3.493670,30.279646,3.888311
min,16.000000,0.000000,0.000000,0.000000
25%,45.000000,3.000000,7.000000,3.000000
50%,58.000000,7.000000,31.000000,6.000000
75%,72.000000,9.000000,65.000000,9.000000
max,152.000000,17.000000,152.000000,31.000000


####  Combine columns into new features

**Extend the price-diff idea to peak, mid-peak, and a shorter 6-month window**

The colleague's feature only looked at the off-peak tariff over a full year. Price sensitivity might
show up more clearly over a shorter, more recent window, and clients also pay peak/mid-peak tariffs —
so we compute the same Dec-vs-Jan-style logic for **all three tariffs**, and repeat it over the
**most recent 6 months** of `price_data` as well as the full year.


In [34]:
def price_diff_features(price_df, months_window=None):
    
    tariff_cols = ['price_off_peak_var', 'price_peak_var', 'price_mid_peak_var',
                   'price_off_peak_fix', 'price_peak_fix', 'price_mid_peak_fix']

    pdf = price_df.copy()
    if months_window is not None:
        cutoff = pdf['price_date'].max() - pd.DateOffset(months=months_window)
        pdf = pdf[pdf['price_date'] > cutoff]

    monthly = pdf.groupby(['id', 'price_date'])[tariff_cols].mean().reset_index()
    first = monthly.groupby('id').first()[tariff_cols]
    last = monthly.groupby('id').last()[tariff_cols]

    suffix = f'_{months_window}m' if months_window else '_year'
    out = (last - first).add_prefix('diff').add_suffix(suffix).reset_index()
    return out


year_diffs = price_diff_features(price_df)
six_m_diffs = price_diff_features(price_df, months_window=6)

df = df.merge(year_diffs, on='id', how='left').merge(six_m_diffs, on='id', how='left')
df.filter(regex='^diff').describe()


,diffprice_off_peak_var_year,diffprice_peak_var_year,diffprice_mid_peak_var_year,diffprice_off_peak_fix_year,diffprice_peak_fix_year,diffprice_mid_peak_fix_year,diffprice_off_peak_var_6m,diffprice_peak_var_6m,diffprice_mid_peak_var_6m,diffprice_off_peak_fix_6m,diffprice_peak_fix_6m,diffprice_mid_peak_fix_6m
count,14606.000000,14606.000000,14606.000000,14606.000000,14606.000000,14606.000000,14606.000000,14606.000000,14606.000000,14606.000000,14606.000000,14606.000000
mean,-0.004566,-0.000827,0.000480,0.278495,-0.046053,-0.025828,-0.005576,-0.001307,0.000166,0.159687,0.008164,0.011017
std,0.012049,0.010844,0.007119,1.349231,2.186263,1.423582,0.007682,0.009562,0.005960,1.119174,1.887067,1.232266
min,-0.148477,-0.143815,-0.098720,-44.266931,-36.344726,-16.724389,-0.149161,-0.109324,-0.096576,-44.444710,-36.490689,-16.724391
25%,-0.007829,-0.002904,0.000000,0.000004,0.000000,0.000000,-0.008161,-0.004169,0.000000,0.000000,0.000000,0.000000
50%,-0.005344,0.000000,0.000000,0.162916,0.000000,0.000000,-0.004628,0.000000,0.000000,0.000000,0.000000,0.000000
75%,-0.003520,0.000000,0.000718,0.177779,0.097749,0.065166,-0.004446,0.000000,0.000000,0.177780,0.000000,0.000000
max,0.168958,0.148474,0.103502,40.728885,36.490689,16.791555,0.123886,0.138103,0.099232,15.177780,36.490689,16.791555


** Other combined features**

- `consumption_growth`: forecast vs. actual usage over the last 12 months — a large *negative* value
  (forecast much lower than actual) could flag a client planning to scale down or leave
- `power_to_consumption_ratio`: subscribed power relative to actual usage — clients paying for far
  more capacity than they use are natural churn candidates
- `has_forecast_discount`: `forecast_discount_energy` is 0 for ~96% of clients, so *whether* a client
  has any forecast discount matters more than its exact size
- `tenure_to_end_ratio`: how far through the visible part of the relationship a client is
  (time since activation vs. time left on the contract)
- `margin_per_product`: net margin spread across active products, to separate "high margin from one
  product" from "high margin because of many products"


In [35]:
df['consumption_growth'] = df['forecast_cons_12m'] - df['cons_12m']

df['power_to_consumption_ratio'] = df['pow_max'] / (df['cons_12m'] + 1)

df['has_forecast_discount'] = (df['forecast_discount_energy'] > 0).astype(int)

df['tenure_to_end_ratio'] = df['months_activ'] / (df['months_activ'] + df['months_to_end'] + 1)

df['margin_per_product'] = df['net_margin'] / (df['nb_prod_act'] + 1)

df[['consumption_growth', 'power_to_consumption_ratio', 'has_forecast_discount',
    'tenure_to_end_ratio', 'margin_per_product']].describe()


,consumption_growth,power_to_consumption_ratio,has_forecast_discount,tenure_to_end_ratio,margin_per_product
count,1.460600e+04,14606.000000,14606.000000,14606.000000,14606.000000
mean,-1.573517e+05,0.290839,0.035054,0.884416,84.986644
std,5.730070e+05,4.723907,0.183923,0.056721,120.658283
min,-6.207019e+06,0.000002,0.000000,0.632653,0.000000
25%,-3.746204e+04,0.000462,0.000000,0.847059,22.607083
50%,-1.260576e+04,0.001117,0.000000,0.890411,50.588333
75%,-4.961927e+03,0.002490,0.000000,0.929412,109.013750
max,3.534240e+03,260.000000,1.000000,0.993103,6142.662500


#### 4. Encode the remaining categorical columns

- `has_gas`: `t`/`f` → `1`/`0`
- `channel_sales` / `origin_up`: one-hot encoded, folding any category with fewer than 100 clients
  into `"other"` first so rare codes don't each get their own near-empty column


In [36]:
df['has_gas'] = df['has_gas'].map({'t': 1, 'f': 0})

MIN_CATEGORY_COUNT = 100
for col in ['channel_sales', 'origin_up']:
    counts = df[col].value_counts()
    rare = counts[counts < MIN_CATEGORY_COUNT].index
    df[col] = df[col].where(~df[col].isin(rare), other='other')

df = pd.get_dummies(df, columns=['channel_sales', 'origin_up'], prefix=['channel', 'origin'])
dummy_cols = [c for c in df.columns if c.startswith('channel_') or c.startswith('origin_')]
df[dummy_cols] = df[dummy_cols].astype(int)

df.shape


(14606, 64)

#### 5. Final check and export

Confirm the table is fully numeric (aside from `id`) with no missing values, then save it ready for modeling.


In [37]:
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Non-numeric columns (should only be 'id'):", df.select_dtypes(include='object').columns.tolist())

df.to_csv('data_for_predictions.csv', index=False)
df.head(3)


Shape: (14606, 64)
Missing values: 0
Non-numeric columns (should only be 'id'): ['id']


,id,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,...,channel_MISSING,channel_ewpakwlliwisiwduibdlfmalxowmwpci,channel_foosdfpfkusacimwkcsosbicdxkicaua,channel_lmkebamcaaclubfxadlmueccxoimlema,channel_other,channel_usilxuppasemubllopkaafesmlibmsdf,origin_kamkkxfxxuwbdslkwifmmcsiusiuosws,origin_ldkssxwpmemidmecebumciepifcamkci,origin_lxidpiddsbxsbosboudacockeimpuepw,origin_other
0,24011ae4ebbe3035111d65fa7c15bc57,0,54946,0,0.00,0,0.0,1.78,0.114481,0.098142,...,0,0,1,0,0,0,0,0,1,0
1,d29c2c54acc38ff3c0614d0a653813dd,4660,0,0,189.95,0,0.0,16.27,0.145711,0.000000,...,1,0,0,0,0,0,1,0,0,0
2,764c75f661154dac3a6c254cd082ea7d,544,0,0,47.96,0,0.0,38.72,0.165794,0.087899,...,0,0,1,0,0,0,1,0,0,0
